# Final Pipeline Persistence with Pickle

This notebook persists the final fitted SECOM pipeline using Python `pickle` and validates that loading it produces identical predictions and Fail probabilities.

## Contents

1. [Goal](#1-goal)
2. [Data and Final Pipeline](#2-data-and-final-pipeline)
3. [Fit and Save Pipeline](#3-fit-and-save-pipeline)
4. [Round-Trip Validation](#4-round-trip-validation)
5. [Persistence Notes](#5-persistence-notes)

## 1. Goal

The saved artifact should contain the complete fitted preprocessing-and-model pipeline, not only the classifier. This includes:

- median imputer;
- constant-feature filter;
- supervised feature selector;
- Random Forest classifier.

The final selected model is `RF + SelectKBest tuned`, consistent with the main notebook's conclusion.

In [1]:
# Imports
from pathlib import Path
import pickle
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import sklearn
import imblearn
import mlflow

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from src.evaluation import get_fail_probabilities, summarize_model

## 2. Data and Final Pipeline

The same stratified train/test split is used for consistency with the main notebook. The pipeline is then fitted on the full training set only.

In [2]:
data_path = PROJECT_ROOT / "data" / "raw" / "secom.data"
labels_path = PROJECT_ROOT / "data" / "raw" / "secom_labels.data"

X = pd.read_csv(data_path, sep=r"\s+", header=None, na_values="NaN")
y = pd.read_csv(labels_path, sep=r"\s+", header=None, names=["target", "timestamp"])["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

best_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("variance_filter", VarianceThreshold(threshold=0.0)),
    ("feature_selector", SelectKBest(score_func=f_classif, k=200)),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        max_depth=8,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    )),
])

final_threshold = 0.40

print(f"Training shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Training shape: (1253, 590)
Test shape: (314, 590)


## 3. Fit and Save Pipeline

The full fitted pipeline is saved with the highest pickle protocol under `models/secom_best_pipeline.pkl`.

In [3]:
models_dir = PROJECT_ROOT / "models"
models_dir.mkdir(parents=True, exist_ok=True)
model_path = models_dir / "secom_best_pipeline.pkl"

best_pipeline.fit(X_train, y_train)

pred_before = np.where(
    get_fail_probabilities(best_pipeline, X_test) >= final_threshold,
    1,
    -1,
)
proba_before = get_fail_probabilities(best_pipeline, X_test)

with model_path.open("wb") as file:
    pickle.dump(best_pipeline, file, protocol=pickle.HIGHEST_PROTOCOL)

model_size_mb = model_path.stat().st_size / (1024 * 1024)
print(f"Saved model to: {model_path}")
print(f"Pickle protocol used: {pickle.HIGHEST_PROTOCOL}")
print(f"Model file size: {model_size_mb:.2f} MB")

Saved model to: E:\SECOM_Dataset_proj\models\secom_best_pipeline.pkl
Pickle protocol used: 5
Model file size: 0.56 MB


## 4. Round-Trip Validation

The validation checks that predictions and Fail probabilities are identical after loading the pickle file.

In [4]:
with model_path.open("rb") as file:
    loaded_pipeline = pickle.load(file)

pred_after = np.where(
    get_fail_probabilities(loaded_pipeline, X_test) >= final_threshold,
    1,
    -1,
)
proba_after = get_fail_probabilities(loaded_pipeline, X_test)

assert np.array_equal(pred_before, pred_after)
assert np.allclose(proba_before, proba_after)

round_trip_metrics = summarize_model(
    f"RF + SelectKBest tuned threshold {final_threshold:.2f}",
    y_test,
    pred_after,
    proba_after,
)

print("Round-trip validation passed.")
round_trip_metrics

Round-trip validation passed.


{'model': 'RF + SelectKBest tuned threshold 0.40',
 'accuracy': 0.9012738853503185,
 'balanced_accuracy': 0.6376564277588168,
 'fail_precision': 0.2916666666666667,
 'fail_recall': 0.3333333333333333,
 'fail_f1': 0.3111111111111111,
 'roc_auc': 0.7739314155696407,
 'pr_auc': 0.2835000855202607}

## 5. Persistence Notes

The pickle file contains the fitted preprocessing and model steps together, so inference uses the same imputation, filtering, feature selection, and classifier learned from the training data.

Security and compatibility notes:

- pickle files should only be loaded from trusted sources;
- loading a pickle can execute code during deserialization;
- the loading environment should use compatible Python and library versions;
- the model was created for this project environment, not as a portable production artifact.

In [5]:
environment_versions = pd.DataFrame([
    {"package": "python", "version": sys.version.split()[0]},
    {"package": "numpy", "version": np.__version__},
    {"package": "pandas", "version": pd.__version__},
    {"package": "scikit-learn", "version": sklearn.__version__},
    {"package": "imbalanced-learn", "version": imblearn.__version__},
    {"package": "mlflow", "version": mlflow.__version__},
])

environment_path = PROJECT_ROOT / "environment_versions.txt"
with environment_path.open("w", encoding="utf-8") as file:
    for row in environment_versions.itertuples(index=False):
        file.write(f"{row.package}=={row.version}\n")

environment_versions

,package,version
0,python,3.13.7
1,numpy,2.4.6
2,pandas,2.3.3
3,scikit-learn,1.9.0
4,imbalanced-learn,0.14.2
5,mlflow,3.14.0
